In [1]:
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader,TensorDataset
import matplotlib.pyplot as plt

In [2]:
df=pd.read_csv("powerplant_data.csv")

In [3]:
df.head(3)

,AT,V,AP,RH,PE
0,8.34,40.77,1010.84,90.01,480.48
1,23.64,58.49,1011.40,74.20,445.75
2,29.74,56.90,1007.15,41.91,438.76


In [7]:
df=df.rename(
    {"AT":"Temperature",
     "V":"Velocity",
     "AP":"Pressure",
     "RH":"Humidity",
     "PE":"Produced_Energy"},axis=1
     )

In [8]:
df.head(2)

,Temperature,Velocity,Pressure,Humidity,Produced_Energy
0,8.34,40.77,1010.84,90.01,480.48
1,23.64,58.49,1011.40,74.20,445.75


In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9568 entries, 0 to 9567
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Temperature      9568 non-null   float64
 1   Velocity         9568 non-null   float64
 2   Pressure         9568 non-null   float64
 3   Humidity         9568 non-null   float64
 4   Produced_Energy  9568 non-null   float64
dtypes: float64(5)
memory usage: 373.9 KB


In [10]:
df.isnull().sum()

Temperature        0
Velocity           0
Pressure           0
Humidity           0
Produced_Energy    0
dtype: int64

In [11]:
X=df.drop("Produced_Energy",axis=1)
y=df["Produced_Energy"]

## Train-Test split

In [14]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

## Scaling the data

In [16]:
scaler=StandardScaler()

X_train_scaled=scaler.fit_transform(X_train)
X_test_scaled=scaler.transform(X_test)

In [19]:
type(X_train_scaled)

numpy.ndarray

In [18]:
type(y_train)

pandas.core.series.Series

In [20]:
y_train.shape

(7654,)

## Pytorch Tensor

In [21]:
X_train_tensor=torch.tensor(X_train_scaled,dtype=torch.float32)
y_train_tensor=torch.tensor(y_train.values,dtype=torch.float32).view(-1,1)

X_test_tensor=torch.tensor(X_test_scaled,dtype=torch.float32)
y_test_tensor=torch.tensor(y_test.values,dtype=torch.float32).view(-1,1)

# Since y_train and y_test are not scaled and are currently pandas series, and X_train_scaled, X_test_scaled are numpy array, there is a type difference hence we use .values to extract values.
# .view is used to change dimensions, currently the dimension is (n,) but pytorch would require (n,1), where -1 is the total rows, 1 is the number of cols

## Tensor Dataset and Data Loader

In [22]:
train_dataset=TensorDataset(X_train_tensor,y_train_tensor)

test_dataset=TensorDataset(X_test_tensor,y_test_tensor)

In [23]:
train_loader=DataLoader(train_dataset,batch_size=32,shuffle=True)

test_loader=DataLoader(test_dataset,batch_size=32)

## Deep Learning

### Define the ANN model

In [29]:
class ANN(nn.Module):
    def __init__(self):
        super(ANN,self).__init__()

        self.model=nn.Sequential(
            # 1st Hidden Layer
            nn.Linear(X_train.shape[1],6),
            nn.ReLU(),

            # 2nd Hidden Layer
            nn.Linear(6,6),
            nn.ReLU(),

            # Output Layer
            nn.Linear(6,1)
         
        )   

    def forward(self,x):
        return self.model(x)

In [30]:
model=ANN()